# 05 · pandas 表格数据

用 pandas 把列与栏组成的表格数据（tabular data）加载、查看、筛选与聚合，创建数据分析的基本工作流。

## 学习目标
- 能用多种方式创建数据框（DataFrame），并理解列索引（index）与栏（columns）的结构。
- 能读取常见文件格式（CSV、Parquet）并快速查看数据的形状与类型。
- 能依条件选取字段与过滤列，组合出想要的子表。
- 能用次数统计（value_counts）与分组聚合（groupby aggregation）做摘要分析。
- 能辨识并处理缺失值（NaN, Not a Number）。
- 能在 pandas 与 NumPy 数组之间互相转换。

## 创建 DataFrame 与基本结构

数据框（DataFrame）是「有标签的二维表格」：每一列（row）有列索引（index），每一栏（column）有栏名。
单独一栏是一个串行（Series），DataFrame 可看成多个共用同一组 index 的 Series 并排。

为什么学：列与栏各自有名字，后续所有选取、过滤、聚合都靠这些标签定位，这是整个 pandas 的基础。

最常见的创建方式是由字典（dict）创建，键（key）变栏名，值（list）变整栏数据。

In [ ]:
# 概念：由字典 dict 创建 DataFrame，每个 key 变成一栏
import pandas as pd

# 造一组假的学生数据当示范用
students = {
    'name': ['小明', '小华', '小美', '阿强'],   # 姓名栏
    'score': [85, 72, 91, 68],                 # 分数栏
    'class': ['A', 'B', 'A', 'B'],             # 班级栏
}
df = pd.DataFrame(students)

print(df)
print('列索引 index：', list(df.index))   # 预设是 0,1,2,... 的整数索引
print('栏名 columns：', list(df.columns))
print('单独取一栏是 Series，类型为：', type(df['score']))

### 由嵌套清单创建

若数据是「一列一笔」的嵌套清单（list of lists），可直接传入并另外指定栏名。
两种方式结果相同，选哪种看数据源较顺手。

In [ ]:
# 概念：由嵌套清单 list 创建，每个子 list 是一列，columns 指定栏名
rows = [
    ['小明', 85, 'A'],
    ['小华', 72, 'B'],
    ['小美', 91, 'A'],
]
df2 = pd.DataFrame(rows, columns=['name', 'score', 'class'])

print(df2)
# 技巧：也能自订 index 取代预设整数索引，让列也有名字
df3 = pd.DataFrame(rows, columns=['name', 'score', 'class'], index=['s1', 's2', 's3'])
print(df3)

## 加载数据档

实务数据多半来自文件。先掌握「写出再读回」的最小循环：`to_csv` / `to_parquet` 写出，`read_csv` / `read_parquet` 读回。

两种格式差异：

| 格式 | 性质 | 特点 |
|---|---|---|
| CSV | 纯文字 | 人类可读、通用，但无类型信息、文件较大 |
| Parquet | 栏式二进位 | 保留类型、压缩率高、读取快，但非纯文字 |

本范例用程序自造的暂存盘示范，全程不依赖任何外部文件。

In [ ]:
# 概念：写出再读回的最小循环（先 to_csv / to_parquet，再 read_csv / read_parquet）
import os
import tempfile

df = pd.DataFrame({
    'name': ['小明', '小华', '小美'],
    'score': [85, 72, 91],
})

# 用系统暂存文件夹，避免在项目留下文件
tmp_dir = tempfile.gettempdir()
csv_path = os.path.join(tmp_dir, 'demo_students.csv')

df.to_csv(csv_path, index=False)        # 注意：index=False 才不会把列索引也写进文件
back_csv = pd.read_csv(csv_path)        # 读回成新的 DataFrame
print('CSV 读回：')
print(back_csv)

# 技巧：Parquet 需要额外引擎（pyarrow），环境若没装就略过，不影响其他单元
try:
    pq_path = os.path.join(tmp_dir, 'demo_students.parquet')
    df.to_parquet(pq_path)
    back_pq = pd.read_parquet(pq_path)
    print('Parquet 读回（类型被保留）：')
    print(back_pq.dtypes)
except Exception as e:
    print('Parquet 未激活（缺 pyarrow）：', e)

## 查看与摘要数据

拿到一份新数据的标准第一步：先看大小、前几列、栏名与每栏类型（dtype），再决定下一步。

| 属性 / 方法 | 回传 |
|---|---|
| `.shape` | （列数, 栏数）的元组 |
| `.head(n)` | 前 n 列（预设 5） |
| `.columns` | 栏名 |
| `.dtypes` | 每栏的数据类型（dtype） |
| `.to_numpy()` | 把整张表降为 NumPy 数组 |

In [ ]:
# 概念：查看一份新数据的标准四件事（shape / head / columns / dtypes）
import numpy as np

# 造一份多栏、较多列的假数据
df = pd.DataFrame({
    'city': ['台北', '台中', '高雄', '台北', '台南', '台中'],
    'population': [264, 281, 277, 264, 187, 281],   # 万人
    'area_km2': [271.8, 2214.9, 2951.9, 271.8, 2191.7, 2214.9],
})

print('形状（列, 栏）：', df.shape)
print('前 3 列：')
print(df.head(3))
print('栏名：', list(df.columns))
print('每栏类型：')
print(df.dtypes)
print('转成 NumPy 数组（会混合类型变 object）：')
print(df.to_numpy())

## 字段选取与条件过滤

分析常从「只看我关心的栏与符合条件的列」开始。

- 菜单栏：`df['栏名']`（得到 Series）。
- 选多栏：`df[['栏A', '栏B']]`（传入栏名 list，得到 DataFrame）。
- 过滤列：先用比较运算造出布尔遮罩（boolean mask），再把遮罩放回 `df[...]` 只留下 True 的列。

形状：`df[df['栏名'] > 门槛]`

In [ ]:
# 概念：布尔遮罩 boolean mask 过滤列，再选取想要的栏
df = pd.DataFrame({
    'name': ['小明', '小华', '小美', '阿强', '小芳'],
    'score': [85, 72, 91, 68, 88],
    'class': ['A', 'B', 'A', 'B', 'A'],
})

mask = df['score'] > 80          # 对每一列做比较，得到一整栏 True/False
print('布尔遮罩：')
print(mask.tolist())

high = df[mask]                  # 只留下遮罩为 True 的列
print('分数大于 80 的列：')
print(high)

# 技巧：过滤与选栏可一气呵成；多条件用 & | 并各自加括号
result = df[(df['score'] > 80) & (df['class'] == 'A')][['name', 'score']]
print('A 班且分数大于 80，只看姓名与分数：')
print(result)

## 次数统计与分组聚合

把细项数据压缩成有意义的摘要，是表格分析的核心。

- `value_counts()`：数单栏各类别出现几次。
- `groupby('栏')`：依某栏的类别分组，再对各组套用聚合函数（aggregation）如 `mean`、`sum`、`count`。

形状：`df.groupby('分组栏')['数值栏'].mean()`

In [ ]:
# 概念：value_counts 数次数，groupby 分组后聚合 aggregation
df = pd.DataFrame({
    'name': ['小明', '小华', '小美', '阿强', '小芳', '阿明'],
    'score': [85, 72, 91, 68, 88, 75],
    'class': ['A', 'B', 'A', 'B', 'A', 'B'],
})

print('各班级出现次数：')
print(df['class'].value_counts())

print('各班级平均分数：')
print(df.groupby('class')['score'].mean())

# 技巧：用 agg 一次求多个统计量，栏名清楚
summary = df.groupby('class')['score'].agg(['mean', 'count', 'max'])
print('各班级摘要（平均、人数、最高分）：')
print(summary)

## 缺失值处理

真实数据常有空缺，pandas 用缺失值（NaN, Not a Number）表示。

1. 侦测：`isna()` 回传布尔遮罩，标出哪里是缺值。
2. 丢弃：`dropna()` 直接移除含缺值的列。
3. 填补：`fillna(值)` 用合理的值（如中位数、0）补上。

为什么学：缺值会污染统计（如平均），必须先决定丢弃或填补。

In [ ]:
# 概念：缺失值 NaN 的侦测 isna、丢弃 dropna、填补 fillna
# 注意：用 np.nan 在数值栏刻意制造缺值
df = pd.DataFrame({
    'name': ['小明', '小华', '小美', '阿强'],
    'score': [85, np.nan, 91, np.nan],
})

print('侦测缺值（每格是否为 NaN）：')
print(df['score'].isna().tolist())

print('丢弃含缺值的列：')
print(df.dropna())

# 技巧：填补前先算未缺值的中位数，避免被 NaN 影响（中位数本身会自动略过 NaN）
median_score = df['score'].median()
filled = df.fillna({'score': median_score})
print('用中位数', median_score, '填补后：')
print(filled)

## DataFrame 与 NumPy 互转

当需要对整批数值做矢量化运算时，常把表格降为纯 NumPy 数组计算，算完再加回栏名变回表格。

- 表格转数组：`df.to_numpy()` 或 `df[['栏...']].to_numpy()`。
- 数组转表格：`pd.DataFrame(数组, columns=[...])`。

取舍：纯数组计算快但失去栏名与类型信息；加回栏名才好阅读与后续分析。

In [ ]:
# 概念：DataFrame 转 NumPy 做矢量化运算，再包回带栏名的 DataFrame
df = pd.DataFrame({
    'math': [80, 65, 90],
    'english': [70, 88, 60],
})

arr = df.to_numpy()              # 取出纯数值数组（无栏名）
print('降为数组：')
print(arr)

# 矢量化：整批同时加 5 分（广播 broadcasting），不需循环
adjusted = arr + 5

# 把运算结果包回 DataFrame，重新粘贴栏名
result = pd.DataFrame(adjusted, columns=df.columns)
print('整体加 5 分后重新变回表格：')
print(result)
# 技巧：也可直接 df + 5，pandas 会保留栏名；此处示范「降阶再升阶」的工作流
print('每位学生两科总分（用数组沿栏方向相加）：', arr.sum(axis=1))

## 练习

以下三题由简到难，皆为复合型但简短。数据自己用字典 / list / numpy 造（仿真即可），不引用任何外部文件。只需在 `# TODO` 处补上程序。

In [ ]:
# TODO 1 ·（简单）建筑基本数据表（集成：创建 DataFrame + 字段选取与条件过滤）
#   情境：手上有几栋建筑的楼高 height（公尺）与楼层数 floors，想挑出高楼。
#   要求：
#     1. 用字典 dict 自造一个小 DataFrame，字段为 building_id、height、floors，
#        数据用 list 直接写几笔仿真数字即可。
#     2. 用布尔遮罩 boolean mask 过滤出 height 大于 30 的列，
#        并只选取 building_id 与 height 两栏。
#   验收：应该看到一张只剩高楼、且只有两栏的子表。
# TODO: 在这里完成


In [ ]:
# TODO 2 ·（中间）用途分类的容积摘要
#   （集成：创建 DataFrame + 缺失值处理 + value_counts + groupby 聚合）
#   情境：一批建筑有用途标签 usage（住宅 / 商业 / 工业）与楼地板面积 gfa（gross floor area），
#        部分 gfa 缺漏，想看各用途的规模摘要。
#   要求：
#     1. 自造含 usage 栏与 gfa 栏的 DataFrame，并刻意让几笔 gfa 为缺失值（用 np.nan）。
#     2. 用 isna 侦测缺值后，以中位数或 0 等合理策略用 fillna 填补。
#     3. 用 value_counts 数各 usage 出现的次数。
#     4. 用 groupby 依 usage 聚合，求各类的平均 gfa 与栋数（count）。
#   验收：应该看到一张以 usage 为列、含平均 gfa 与栋数的摘要表，且数据不再有缺值。
# TODO: 在这里完成


In [ ]:
# TODO 3 ·（稍难）街廓网格的容积率前后比较
#   （集成：创建 DataFrame + DataFrame 与 NumPy 互转 + 条件过滤 + groupby 聚合）
#   情境：每栋建筑属于某个街廓网格 grid，已知占地面积 footprint 与楼地板面积 gfa；
#        要评估一个「高度乘数」政策情境下，各网格的容积率 FAR（floor area ratio）如何变化。
#   要求：
#     1. 自造含 grid、footprint、gfa 三栏的 DataFrame。
#     2. 用 .to_numpy() 取出 gfa 与 footprint 两数值栏，矢量化算现况 FAR = gfa / footprint；
#        再对 gfa 乘上高度乘数（如 1.2）算政策后 FAR。
#     3. 把现况 FAR 与政策后 FAR 两结果包回带栏名的 DataFrame（连同 grid 栏）。
#     4. 过滤出政策后 FAR 超过上限（如 3.0）的列，并用 groupby 依 grid 聚合，
#        求各网格政策后 FAR 的平均与超标栋数。
#   验收：应该看到一张各网格的聚合表，能看出哪些网格政策后平均容积率偏高、有几栋超标。
# TODO: 在这里完成


## 小结

- DataFrame 是有列索引（index）与栏名的二维表格，单栏是 Series；可由字典或嵌套清单创建。
- 用 `to_csv` / `read_csv`、`to_parquet` / `read_parquet` 完成「写出再读回」的文件循环；CSV 为纯文字、Parquet 为栏式二进位。
- 拿到新数据先看 `.shape`、`.head()`、`.columns`、`.dtypes`。
- 用布尔遮罩过滤列、用栏名 list 选多栏，组出想要的子表。
- `value_counts` 数次数，`groupby` 分组后用聚合函数做摘要。
- 缺失值（NaN）用 `isna` 侦测、`dropna` 丢弃、`fillna` 填补。
- 用 `.to_numpy()` 把表格降为数组做矢量化运算，再包回带栏名的 DataFrame。